In [ ]:
"""
Scrap Welcome to the Jungle
extraction des liens d'annonces
MULTITHREADING ET DÉTECTION DE COMPÉTENCES
"""

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import TimeoutException, NoSuchElementException
import time
import json
import os
import re
from datetime import datetime
from urllib.parse import urlparse, parse_qs, urlencode, urlunparse
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Lock


def detecter_competences_stricte(texte):
    """
    Cherche uniquement des correspondances exactes de mots complets
    Args:
        texte: Le texte dans lequel chercher les compétences
    Returns:
        Liste des compétences trouvées
    """
    if not texte:
        return []
    
    # Charger les compétences depuis le fichier
    try:
        from competences_techniques import TOUTES_COMPETENCES
    except ImportError:
        print("Fichier competences_techniques.py non trouvé, détection désactivée")
        return []
    
    texte_lower = texte.lower()
    competences_trouvees = set()
    
    for competence in TOUTES_COMPETENCES:
        # Ignorer les compétences trop courtes (1-2 lettres) sauf exceptions
        if len(competence) <= 2 and competence not in ['GO', 'C#', '.NET']:
            continue
        
        competence_lower = competence.lower()
        
        # RECHERCHE STRICTE : Chercher uniquement avec word boundaries
        # \b assure qu'on cherche un mot complet, pas une sous-chaîne
        pattern = r'\b' + re.escape(competence_lower) + r'\b'
        
        if re.search(pattern, texte_lower, re.IGNORECASE):
            competences_trouvees.add(competence)
        # Cas spéciaux pour les technologies avec points ou tirets
        # Exemple : "React.js" pourrait être écrit "React" dans le texte
        elif '.' in competence_lower or '-' in competence_lower:
            # Essayer aussi la version sans le suffixe
            base_name = competence_lower.split('.')[0].split('-')[0]
            if len(base_name) > 2:  # Éviter les faux positifs sur des noms courts
                pattern_base = r'\b' + re.escape(base_name) + r'\b'
                match = re.search(pattern_base, texte_lower, re.IGNORECASE)
                if match:
                    # Vérifier que le contexte est valide
                    start = match.start()
                    end = match.end()
                    
                    # S'assurer qu'il n'y a pas de lettres juste avant ou après
                    char_before = texte_lower[start-1] if start > 0 else ' '
                    char_after = texte_lower[end] if end < len(texte_lower) else ' '
                    
                    if not (char_before.isalpha() or char_after.isalpha()):
                        competences_trouvees.add(competence)
    
    return sorted(list(competences_trouvees))

class WTTJScrap:
    def __init__(self, headless=True, max_workers=5):
        """Initialise le scraper avec Selenium"""
        chrome_options = Options()
        if headless:
            chrome_options.add_argument('--headless')
        chrome_options.add_argument('--no-sandbox')
        chrome_options.add_argument('--disable-dev-shm-usage')
        chrome_options.add_argument('--disable-blink-features=AutomationControlled')
        chrome_options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36')
        
        self.chrome_options = chrome_options
        self.driver = webdriver.Chrome(options=chrome_options)
        self.driver.maximize_window()
        self.jobs = []
        self.seen_urls = set()
        self.max_workers = max_workers
        self.lock = Lock()  # Pour le multithreading
    
    def _create_driver(self):
        """Crée une nouvelle instance de driver pour les threads"""
        return webdriver.Chrome(options=self.chrome_options)
    
    def scroll_page(self, driver=None, pause_time=2):
        """Scroll progressif pour charger toutes les annonces"""
        if driver is None:
            driver = self.driver
            
        last_height = driver.execute_script("return document.body.scrollHeight")
        
        for _ in range(3):  # Faire 3 scrolls
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(pause_time)
            new_height = driver.execute_script("return document.body.scrollHeight")
            if new_height == last_height:
                break
            last_height = new_height
    
    def extract_job_description(self, driver, job_url):
        """
        Visite l'URL d'une annonce et extrait la description complète
        
        Args:
            driver: Instance de WebDriver
            job_url: URL de l'annonce à visiter
            
        Returns:
            str: Description complète de l'offre
        """
        try:
            driver.get(job_url)
            time.sleep(2)
            """
            # Chercher et cliquer sur TOUS les boutons "Voir plus" s'ils existent
            # Il peut y en avoir plusieurs sur la page
            try:
                # Sélecteur principal : data-testid="view-more-btn"
                voir_plus_buttons = driver.find_elements(By.CSS_SELECTOR, "[data-testid='view-more-btn']")
                
                if voir_plus_buttons:
                    print(f" {len(voir_plus_buttons)} bouton(s) 'Voir plus' trouvé(s)")
                    
                    for idx, voir_plus in enumerate(voir_plus_buttons, 1):
                        try:
                            # Vérifier si le bouton est visible et cliquable
                            if voir_plus.is_displayed():
                                # Scroller vers le bouton
                                driver.execute_script("arguments[0].scrollIntoView(true);", voir_plus)
                                time.sleep(0.5)
                                
                                # Essayer de cliquer
                                try:
                                    voir_plus.click()
                                    print(f"      ✓ Bouton {idx} cliqué")
                                except:
                                    # Si clic normal échoue, utiliser JavaScript
                                    driver.execute_script("arguments[0].click();", voir_plus)
                                    print(f"      ✓ Bouton {idx} cliqué (JS)")
                                
                                time.sleep(1)  # Attendre que le contenu se charge
                        except Exception as e:
                            print(f"      ⚠️ Erreur bouton {idx}: {str(e)[:30]}")
                            continue
                else:
                    print(f"      ℹ️ Pas de bouton 'Voir plus'")
                    
            except Exception as e:
                print(f"      ⚠️ Erreur recherche boutons: {str(e)[:50]}")
            """
            # Extraire la description complète
            description = ""
            description_selectors = [
                # Sélecteurs spécifiques identifiés
                "div[data-is-view-more='true']",
                "div[data-is-text-too-long]",
                "div.sc-dBueTE.jxIZll",
                "div.sc-jHEWpz.kgAAkN",
                
                # Sélecteurs génériques en fallback
                "[data-testid='job-description']",
                "[class*='JobDescription']",
                "[class*='job-description']",
                "section[class*='description']",
                "div[class*='description']",
                "[role='article']",
                "article",
                "main"
            ]
            
            for selector in description_selectors:
                try:
                    desc_element = driver.find_element(By.CSS_SELECTOR, selector)
                    description = desc_element.text.strip()
                    if description and len(description) > 100:
                        print(f"      ✓ Description extraite ({len(description)} caractères) via: {selector}")
                        break
                except:
                    continue
            
            # Si toujours pas de description, essayer de combiner plusieurs sections
            if not description or len(description) < 100:
                try:
                    all_desc_divs = driver.find_elements(By.CSS_SELECTOR, "div[data-is-view-more='true'], section")
                    combined_text = "\n\n".join([div.text.strip() for div in all_desc_divs if div.text.strip()])
                    if combined_text and len(combined_text) > len(description):
                        description = combined_text
                        print(f"      ✓ Description combinée extraite ({len(description)} caractères)")
                except:
                    pass
            
            if not description:
                print(f"      ⚠️ Aucune description trouvée")
            
            return description
            
        except Exception as e:
            print(f"      ❌ Erreur extraction description: {str(e)[:50]}")
            return ""
    
    def process_job_details(self, job_data):
        """
        Fonction exécutée en parallèle pour extraire la description et les compétences
        
        Args:
            job_data: Dictionnaire contenant les infos de base avec 'url'
            
        Returns:
            dict: job_data enrichi avec description et compétences
        """
        driver = None
        try:
            # Créer un driver pour ce thread
            driver = self._create_driver()
            
            job_url = job_data.get('url')
            if not job_url:
                return job_data
            
            # Extraire la description
            description = self.extract_job_description(driver, job_url)
            job_data['description'] = description
            
            # Détecter les compétences
            if description:
                try:
                    # Utiliser la fonction de détection stricte intégrée
                    competences = detecter_competences_stricte(description)
                    job_data['competences'] = competences
                    
                    # Debug : afficher un extrait de la description si aucune compétence
                    if not competences and len(description) > 100:
                        print(f"Aucune compétence détectée.)
                        
                except Exception as e:
                    print(f"Erreur détection compétences: {e}")
                    job_data['competences'] = []
            else:
                job_data['competences'] = []
                print(f"Description vide, impossible de détecter les compétences")
            
            return job_data
            
        except Exception as e:
            print(f"Erreur thread: {str(e)[:50]}")
            job_data['description'] = ""
            job_data['competences'] = []
            return job_data
        finally:
            if driver:
                driver.quit()
    
    def scrape_jobs(self, url, max_pages=None, extract_details=True):
        """
        Scrape les annonces depuis une URL en changeant le numéro de page dans l'URL
        
        Args:
            url: URL de base
            max_pages: Nombre max de pages à scraper
            extract_details: Si True, extrait description et compétences (plus lent)
        """
        print(f"🌐 Accès à l'URL: {url}")
        if extract_details:
            print(f"⚡ Mode multithreading activé : {self.max_workers} workers")
        
        # Extraire l'URL de base et vérifier si elle contient déjà un paramètre page
        parsed_url = urlparse(url)
        query_params = parse_qs(parsed_url.query)
        
        # Retirer le paramètre page s'il existe pour le contrôler manuellement
        if 'page' in query_params:
            del query_params['page']
        
        base_url = urlunparse((
            parsed_url.scheme,
            parsed_url.netloc,
            parsed_url.path,
            parsed_url.params,
            '',  # On reconstruit la query après
            parsed_url.fragment
        ))
        
        page_number = 1
        all_basic_jobs = []  # Pour stocker les jobs avant traitement parallèle
        
        # PHASE 1: Collecter les infos de base
        print("\n" + "="*60)
        print("PHASE 1 : COLLECTE DES INFORMATIONS DE BASE")
        print("="*60)
        
        while True:
            if max_pages and page_number > max_pages:
                break
            
            # Construire l'URL avec le numéro de page
            query_params['page'] = [str(page_number)]
            query_string = urlencode(query_params, doseq=True)
            current_url = f"{base_url}?{query_string}"
            
            print(f"\n{'='*50}")
            print(f"📄 PAGE {page_number}")
            print(f"🔗 {current_url}")
            print('='*50)
            
            # Naviguer vers la page
            self.driver.get(current_url)
            time.sleep(3)
            
            # Gérer le cookie consent (seulement à la première page)
            if page_number == 1:
                try:
                    cookie_button = WebDriverWait(self.driver, 5).until(
                        EC.element_to_be_clickable((By.XPATH, "//button[contains(text(), 'Accepter') or contains(text(), 'Accept') or contains(@id, 'accept')]"))
                    )
                    cookie_button.click()
                    print("✅ Cookies acceptés")
                    time.sleep(1)
                except:
                    print("ℹ️  Pas de bannière de cookies")
            
            # Scroll pour charger le contenu
            self.scroll_page()
            time.sleep(2)
            
            # Chercher tous les liens vers des annonces
            job_links = self.driver.find_elements(By.CSS_SELECTOR, "a[href*='/jobs/']")
            
            # Filtrer pour ne garder que les vraies annonces
            valid_links = []
            for link in job_links:
                href = link.get_attribute('href')
                
                # Vérifier que c'est bien un lien d'annonce complet
                if href and '/companies/' in href and '/jobs/' in href and href not in self.seen_urls:
                    # Exclure les liens de navigation/filtres
                    data_testid = link.get_attribute('data-testid') or ''
                    if 'filter' not in data_testid:
                        valid_links.append(link)
                        self.seen_urls.add(href)
            
            print(f"🔗 {len(valid_links)} nouvelles annonces trouvées sur cette page")
            
            if not valid_links:
                print("⚠️  Aucune annonce trouvée - fin du scraping")
                break
            
            # Extraire les informations de chaque lien
            for idx, link in enumerate(valid_links, 1):
                try:
                    job_data = self.extract_job_from_link(link)
                    if job_data:
                        if extract_details:
                            all_basic_jobs.append(job_data)
                        else:
                            self.jobs.append(job_data)
                        print(f"  ✓ {idx}. {job_data.get('title', 'N/A')[:50]} - {job_data.get('company', 'N/A')[:30]}")
                except Exception as e:
                    print(f"  ✗ {idx}. Erreur: {str(e)[:50]}")
            
            # Passer à la page suivante
            page_number += 1
            time.sleep(2)  # Pause entre les pages
        
        print(f"\n{'='*50}")
        print(f"📊 PHASE 1 TERMINÉE: {len(all_basic_jobs) if extract_details else len(self.jobs)} annonces collectées")
        print('='*50)
        
        # PHASE 2: Traitement parallèle des détails (si demandé)
        if extract_details and all_basic_jobs:
            print("\n" + "="*60)
            print("PHASE 2 : EXTRACTION PARALLÈLE DES DESCRIPTIONS")
            print("="*60)
            print(f"⚡ {self.max_workers} threads en parallèle\n")
            
            completed = 0
            total = len(all_basic_jobs)
            
            with ThreadPoolExecutor(max_workers=self.max_workers) as executor:
                # Soumettre tous les jobs
                future_to_job = {
                    executor.submit(self.process_job_details, job_info): job_info 
                    for job_info in all_basic_jobs
                }
                
                # Traiter les résultats au fur et à mesure
                for future in as_completed(future_to_job):
                    completed += 1
                    try:
                        job_data = future.result()
                        if job_data:
                            with self.lock:
                                self.jobs.append(job_data)
                            
                            comp_count = len(job_data.get('competences', []))
                            print(f"  [{completed}/{total}] ✓ {job_data.get('title', 'N/A')[:40]} "
                                  f"({job_data.get('company', 'N/A')[:20]}) - {comp_count} compétences")
                    except Exception as e:
                        print(f"  [{completed}/{total}] ❌ Erreur: {str(e)[:50]}")
        
        print(f"\n{'='*50}")
        print(f"📊 TOTAL: {len(self.jobs)} annonces collectées")
        print('='*50)
        return self.jobs
    
    def extract_job_from_link(self, link_element):
        """Extrait les informations depuis un lien d'annonce"""
        job_data = {}
        
        try:
            # URL de l'annonce
            url = link_element.get_attribute('href')
            job_data['url'] = url

            
            
            # Remonter au conteneur parent pour trouver toutes les infos
            # Le parent peut être un li, div, ou article
            parent = link_element.find_element(By.XPATH, './ancestor::*[self::li or self::article or self::div][1]')
            
            # Texte complet du conteneur
            full_text = parent.text.strip()
            
            # Titre - peut être dans le lien lui-même ou dans un heading
            #title = link_element.text.strip()
            title = None
            for tag in ['h2']:
                try:
                    title_elem = parent.find_element(By.CSS_SELECTOR, tag)
                    title = title_elem.text.strip()
                    if title:
                     break
                except:
                    continue
        
            if not title:
                title = link_element.text.strip()
                # Chercher dans le parent
                """
                for tag in ['h2']:
                    try:
                        title_elem = parent.find_element(By.CSS_SELECTOR, tag)
                        title = title_elem.text.strip()
                        if title:
                            break
                    except:
                        continue
                """

            job_data['title'] = title if title else None
            
            # Entreprise - chercher le logo ou un élément spécifique
            company = None
            try:
                img = parent.find_element(By.CSS_SELECTOR, "img[alt]")
                company = img.get_attribute('alt').strip()
            except:
                pass
            
            # Si pas trouvé via logo, chercher dans le texte
            if not company:
                try:
                    # Chercher un élément avec "company" dans la classe
                    company_elem = parent.find_element(By.CSS_SELECTOR, "[class*='company'], [class*='Company']")
                    company = company_elem.text.strip()
                except:
                    pass
            
            job_data['company'] = company

            #secteur
            sector = None
            try:
                # Chercher l'icône avec name="tag"
                tag_icon = parent.find_element(By.CSS_SELECTOR, "i[name='tag']")
                # Le texte du secteur est dans un span frère
                sector_span = tag_icon.find_element(By.XPATH, './following-sibling::span')
                sector = sector_span.text.strip()
            except:
                try:
                    # Alternative : chercher directement un div contenant l'icône tag
                    sector_container = parent.find_element(By.XPATH, ".//div[.//i[@name='tag']]")
                    sector = sector_container.text.strip()
                except:
                    pass
            
            job_data['sector'] = sector

            
            # Localisation
            location = None
            try:
                # Chercher un élément avec "location" dans la classe ou près d'une icône de lieu
                location_elem = parent.find_element(By.CSS_SELECTOR, "[class*='location'], [class*='Location']")
                location = location_elem.text.strip()
            except:
                # Chercher via icône
                try:
                    icon = parent.find_element(By.CSS_SELECTOR, "i[name='location'], i[name='marker']")
                    location_elem = icon.find_element(By.XPATH, './following-sibling::*[1]')
                    location = location_elem.text.strip()
                except:
                    pass
            
            job_data['location'] = location
            
            # Type de contrat - chercher l'icône contract et le texte suivant
            contract = None
            try:
                # Chercher l'icône avec name="contract"
                icon = parent.find_element(By.CSS_SELECTOR, "i[name='contract']")
                # Le texte est dans un span frère ou parent proche
                contract_container = icon.find_element(By.XPATH, './ancestor::*[contains(@class, "sc-") or contains(@class, "wui")][1]')
                contract = contract_container.text.strip()
            except:
                try:
                    # Alternative : chercher directement un élément contenant "contract" dans la classe
                    contract_elem = parent.find_element(By.CSS_SELECTOR, "[class*='contract']")
                    contract = contract_elem.text.strip()
                except:
                    pass
            
            job_data['contract_type'] = contract
            
            # Salaire - chercher l'icône salary et le texte "Salaire : XXX"
            salary = None
            try:
                # Chercher l'icône avec name="salary"
                salary_icon = parent.find_element(By.CSS_SELECTOR, "i[name='salary']")
                # Le texte "Salaire : " est dans un span proche
                salary_container = salary_icon.find_element(By.XPATH, './ancestor::*[contains(@class, "sc-") or contains(@class, "wui")][1]')
                salary_text = salary_container.text.strip()
                
                # Nettoyer le texte pour garder seulement la valeur
                if salary_text:
                    salary = salary_text.replace('Salaire :', '').replace('Salaire', '').strip()
            except:
                try:
                    # Alternative : chercher un span contenant "Salaire"
                    salary_elem = parent.find_element(By.XPATH, ".//span[contains(text(), 'Salaire')]")
                    salary_text = salary_elem.find_element(By.XPATH, './parent::*').text.strip()
                    salary = salary_text.replace('Salaire :', '').replace('Salaire', '').strip()
                except:
                    pass
            
            job_data['salary'] = salary
            
            # Parser le texte brut si certaines infos manquent
            if full_text and not all([title, company, location]):
                lines = [l.strip() for l in full_text.split('\n') if l.strip()]
                
                if not title and lines:
                    job_data['title'] = lines[0]
                
                if not company and len(lines) > 1:
                    job_data['company'] = lines[1]
                
                if not location:
                    # Chercher une ligne qui ressemble à une localisation
                    for line in lines:
                        if any(city in line for city in ['Paris', 'Lyon', 'France', 'Remote', 'Télétravail']):
                            job_data['location'] = line
                            break
            
            # Métadonnées
            job_data['scraped_at'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
            
        except Exception as e:
            print(f"    ⚠️ Erreur extraction: {e}")
            return None
        
        # Retourner seulement si on a au moins une URL et un titre
        if job_data.get('url') and job_data.get('title'):
            return job_data
        return None
    
    def save_to_json(self, filename='wttj_jobs.json'):
        """Sauvegarde les résultats en JSON"""
        if not self.jobs:
            print("⚠️  Aucune donnée à sauvegarder")
            return None
        
        # Utiliser le répertoire courant au lieu d'un chemin absolu
        import os
        filepath = os.path.join(os.getcwd(), filename)
        
        with open(filepath, 'w', encoding='utf-8') as f:
            json.dump(self.jobs, f, ensure_ascii=False, indent=2)
        print(f"\n💾 {len(self.jobs)} annonces sauvegardées dans {filepath}")
        return filepath
    
    def close(self):
        """Ferme le navigateur"""
        self.driver.quit()


def main():
    """Exemple d'utilisation"""
    
    # ========== CONFIGURATION ==========
    
    # URL avec filtres CDI/CDD pour la France
    url = "https://www.welcometothejungle.com/fr/jobs?refinementList%5Boffices.country_code%5D%5B%5D=FR&refinementList%5Bcontract_type%5D%5B%5D=full_time&refinementList%5Bcontract_type%5D%5B%5D=temporary&query=&page=1"
    
    # Nombre de pages à scraper (None = toutes les pages, ou mettre un nombre comme 3)
    MAX_PAGES = 2  # 👈 CHANGEZ ICI pour tester (ex: 3, 5, 10, ou None pour tout)
    
    # Afficher le navigateur pendant le scraping ? (False = invisible, True = visible)
    HEADLESS = True  # 👈 False pour voir ce qui se passe, True pour mode silencieux (recommandé pour multithread)
    
    # NOUVELLES OPTIONS
    EXTRACT_DETAILS = True  # 👈 True = extrait description + compétences (plus lent), False = infos de base seulement
    MAX_WORKERS = 5  # 👈 Nombre de threads parallèles (3-5 recommandé)
    
    # ===================================
    
    print(f"\n{'='*60}")
    print(f"🚀 DÉMARRAGE DU SCRAPER WELCOME TO THE JUNGLE")
    print(f"📄 Pages à scraper: {MAX_PAGES if MAX_PAGES else 'TOUTES'}")
    print(f"👁️  Mode navigateur: {'VISIBLE' if not HEADLESS else 'INVISIBLE'}")
    if EXTRACT_DETAILS:
        print(f"⚡ Mode multithreading: {MAX_WORKERS} workers")
        print(f"🎯 Extraction: Descriptions + Compétences")
    else:
        print(f"🎯 Extraction: Informations de base uniquement")
    print(f"{'='*60}")
        
    scraper = WTTJScrap(headless=HEADLESS, max_workers=MAX_WORKERS)
    
    try:
        start_time = time.time()
        
        # Scraper les annonces
        jobs = scraper.scrape_jobs(url, max_pages=MAX_PAGES, extract_details=EXTRACT_DETAILS)
        
        # Sauvegarder
        scraper.save_to_json()
        
        elapsed = time.time() - start_time
        
        # Statistiques
        if jobs:
            print(f"\n{'='*60}")
            print(f"📈 STATISTIQUES FINALES")
            print(f"{'='*60}")
            print(f"   ⏱️  Temps d'exécution: {elapsed:.1f} secondes ({elapsed/60:.1f} minutes)")
            print(f"   📊 Total annonces: {len(jobs)}")
            if elapsed > 0:
                print(f"   ⚡ Vitesse: {len(jobs)/elapsed*60:.1f} annonces/minute")
            
            companies = set(job.get('company') for job in jobs if job.get('company'))
            print(f"   🏢 Entreprises uniques: {len(companies)}")
            
            locations = set(job.get('location') for job in jobs if job.get('location'))
            print(f"   📍 Localisations uniques: {len(locations)}")
            
            if EXTRACT_DETAILS:
                # Statistiques sur les compétences
                all_competences = []
                for job in jobs:
                    if job.get('competences'):
                        all_competences.extend(job.get('competences'))
                
                if all_competences:
                    from collections import Counter
                    competences_count = Counter(all_competences)
                    top_competences = competences_count.most_common(10)
                    
                    print(f"   🎯 Compétences uniques détectées: {len(set(all_competences))}")
                    print(f"\n   📊 Top 10 des compétences les plus demandées:")
                    for i, (comp, count) in enumerate(top_competences, 1):
                        print(f"      {i}. {comp}: {count} fois")
                      
            print(f"\n{'='*60}")
            print(f"✅ SCRAPING TERMINÉ AVEC SUCCÈS !")
            print(f"{'='*60}\n")
        else:
            print("\n⚠️ Aucune annonce collectée")
        
    except Exception as e:
        print(f"\n{'='*60}")
        print(f"❌ ERREUR DURANT LE SCRAPING")
        print(f"{'='*60}")
        print(f"{e}")
        import traceback
        traceback.print_exc()
    
    finally:
        scraper.close()
        print("\n🔒 Navigateur fermé\n")

if __name__ == "__main__":
    main()


🚀 DÉMARRAGE DU SCRAPER WELCOME TO THE JUNGLE
📄 Pages à scraper: 2
👁️  Mode navigateur: INVISIBLE
⚡ Mode multithreading: 5 workers
🎯 Extraction: Descriptions + Compétences
🌐 Accès à l'URL: https://www.welcometothejungle.com/fr/jobs?refinementList%5Boffices.country_code%5D%5B%5D=FR&refinementList%5Bcontract_type%5D%5B%5D=full_time&refinementList%5Bcontract_type%5D%5B%5D=temporary&query=&page=1
⚡ Mode multithreading activé : 5 workers

PHASE 1 : COLLECTE DES INFORMATIONS DE BASE

📄 PAGE 1
🔗 https://www.welcometothejungle.com/fr/jobs?refinementList%5Boffices.country_code%5D%5B%5D=FR&refinementList%5Bcontract_type%5D%5B%5D=full_time&refinementList%5Bcontract_type%5D%5B%5D=temporary&page=1
✅ Cookies acceptés
🔗 36 nouvelles annonces trouvées sur cette page
  ✓ 1. Consultant.e Data Alteryx - The Information Lab
  ✓ 2. Tech Lead Data & Analytics (Snowflake, dbt, Tablea - Accor Tech & Digital
  ✓ 3. Senior Account Executive (F/H) - XXII GROUP
  ✓ 4. Producteur catalogues F/H - Auchan
  ✓ 5. Emp